# V2 Projection

This notebook reads `tmp/v2/new_thermal_bbox.csv`, preserves the original CSV timestamp, optionally reads a new timestamp from the source image EXIF data, standardizes the bbox columns, and projects bbox coordinates to the 2D room map.

All v2 outputs are written under `tmp/v2/` and separated by processing stage.

In [1]:
# load project paths and local modules.
from pathlib import Path
from types import SimpleNamespace
import sys

import numpy as np
import pandas as pd
from PIL import Image

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import config_v2
from projection_v2 import run_projection

In [2]:
# define v2 input and output files.
INPUT_CSV = config_v2.V2_RAW_BBOX_CSV
THERMAL_IMAGE_DIR = config_v2.THERMAL_IMAGES_DIR
STANDARDIZED_CSV = config_v2.V2_STANDARDIZED_BBOX_CSV
PROJECTED_CSV = config_v2.V2_PROJECTED_DETECTIONS_CSV
USE_IMAGE_EXIF_TIMESTAMP = config_v2.USE_IMAGE_EXIF_TIMESTAMP

config_v2.ensure_v2_output_dirs()

{
    'input_csv': INPUT_CSV,
    'thermal_image_dir': THERMAL_IMAGE_DIR,
    'use_image_exif_timestamp': USE_IMAGE_EXIF_TIMESTAMP,
    'standardized_csv': STANDARDIZED_CSV,
    'projected_csv': PROJECTED_CSV,
}


{'input_csv': PosixPath('/Users/pck/TUe_AI_ES/2025_Q3/Team_Intership/2026_group_cleanroom/tmp/v2/new_thermal_bbox.csv'),
 'thermal_image_dir': PosixPath('/Users/pck/TUe_AI_ES/2025_Q3/Team_Intership/2026_group_cleanroom/thermal_images'),
 'use_image_exif_timestamp': True,
 'standardized_csv': PosixPath('/Users/pck/TUe_AI_ES/2025_Q3/Team_Intership/2026_group_cleanroom/tmp/v2/input/v2_thermal_bbox_standardized.csv'),
 'projected_csv': PosixPath('/Users/pck/TUe_AI_ES/2025_Q3/Team_Intership/2026_group_cleanroom/tmp/v2/projection/v2_projected_detections.csv')}

In [3]:
# read the new bbox and temperature CSV.
raw_df = pd.read_csv(INPUT_CSV)
print(f'rows: {len(raw_df):,}')
print(f'columns: {list(raw_df.columns)}')
display(raw_df.head())

rows: 237,515
columns: ['filename', 'timestamp', 'class_name', 'box_x0', 'bbox_y0', 'bbox_x1', 'bbox_y1', 'temp_mean_c', 'temp_max_c']


,filename,timestamp,class_name,box_x0,bbox_y0,bbox_x1,bbox_y1,temp_mean_c,temp_max_c
0,IR_63142.jpg,2025/5/1 15:19,Machine,461,298,483,318,21.886790,25.424925
1,IR_63142.jpg,2025/5/1 15:19,Machine,467,256,491,280,22.069757,25.502478
2,IR_63142.jpg,2025/5/1 15:19,Machine,404,248,438,271,21.645502,23.766699
3,IR_63142.jpg,2025/5/1 15:19,Machine,416,212,435,230,22.321835,25.575266
4,IR_63142.jpg,2025/5/1 15:19,Machine,452,207,507,248,23.885866,25.676907


## Timestamp Handling

If the `timestamp` column in `new_thermal_bbox.csv` is already complete and correct, set `USE_IMAGE_EXIF_TIMESTAMP = False` in the setup cell. In that case, the two EXIF timestamp cells below can be skipped.

Do not skip the standardization cell. It renames columns, creates `detection_index`, adds `confidence`, orders the columns, and writes the standardized CSV used by projection.

In [4]:
# read EXIF DateTime from one image; return NA fields when the image is missing or unreadable.
EXIF_DATETIME_TAGS = [306, 36867, 36868]

def read_image_timestamp(image_name, image_dir=THERMAL_IMAGE_DIR):
    image_name = str(image_name).strip()
    image_path = Path(image_dir) / image_name
    result = {
        'image_name': image_name,
        'image_exists': image_path.exists(),
        'image_timestamp_raw': pd.NA,
        'image_timestamp': pd.NaT,
        'image_timestamp_source': pd.NA,
        'image_timestamp_error': pd.NA,
    }
    if not image_name or not image_path.exists():
        return result
    try:
        with Image.open(image_path) as image:
            exif = image.getexif()
            for tag in EXIF_DATETIME_TAGS:
                value = exif.get(tag)
                if value:
                    result['image_timestamp_raw'] = str(value)
                    result['image_timestamp'] = pd.to_datetime(str(value), format='%Y:%m:%d %H:%M:%S', errors='coerce')
                    result['image_timestamp_source'] = f'exif_{tag}'
                    break
    except Exception as exc:
        result['image_timestamp_error'] = str(exc)
    return result

In [5]:
# build one timestamp lookup row per unique filename to avoid repeatedly opening the same image.
if USE_IMAGE_EXIF_TIMESTAMP:
    unique_filenames = pd.Series(raw_df['filename'].dropna().astype(str).str.strip().unique(), name='filename')
    timestamp_lookup = pd.DataFrame([read_image_timestamp(name) for name in unique_filenames])
else:
    unique_filenames = pd.Series(raw_df['filename'].dropna().astype(str).str.strip().unique(), name='filename')
    timestamp_lookup = pd.DataFrame({
        'image_name': unique_filenames,
        'image_exists': pd.NA,
        'image_timestamp_raw': pd.NA,
        'image_timestamp': pd.NaT,
        'image_timestamp_source': pd.NA,
        'image_timestamp_error': pd.NA,
    })

summary = {
    'unique_filenames': len(unique_filenames),
    'images_found': int(timestamp_lookup['image_exists'].fillna(False).sum()) if 'image_exists' in timestamp_lookup else 0,
    'images_missing': int((~timestamp_lookup['image_exists'].fillna(False)).sum()) if 'image_exists' in timestamp_lookup else 0,
    'timestamps_read': int(timestamp_lookup['image_timestamp'].notna().sum()) if 'image_timestamp' in timestamp_lookup else 0,
    'timestamps_na': int(timestamp_lookup['image_timestamp'].isna().sum()) if 'image_timestamp' in timestamp_lookup else 0,
}
summary


{'unique_filenames': 14662,
 'images_found': 9119,
 'images_missing': 5543,
 'timestamps_read': 9119,
 'timestamps_na': 5543}

In [6]:
# standardize column names and preserve both the original CSV timestamp and the image-derived timestamp.
standard_df = raw_df.rename(
    columns={
        'filename': 'image_name',
        'class_name': 'label',
        'box_x0': 'bbox_x0',
    }
).copy()

standard_df['original_timestamp'] = standard_df['timestamp']
if 'timestamp_lookup' in globals():
    standard_df = standard_df.merge(timestamp_lookup, on='image_name', how='left')
else:
    standard_df['image_exists'] = pd.NA
    standard_df['image_timestamp_raw'] = pd.NA
    standard_df['image_timestamp'] = pd.NaT
    standard_df['image_timestamp_source'] = pd.NA
    standard_df['image_timestamp_error'] = pd.NA

if USE_IMAGE_EXIF_TIMESTAMP:
    standard_df['timestamp'] = standard_df['image_timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S')
    standard_df.loc[standard_df['image_timestamp'].isna(), 'timestamp'] = pd.NA
else:
    standard_df['timestamp'] = pd.to_datetime(standard_df['original_timestamp'], errors='coerce').dt.strftime('%Y-%m-%d %H:%M:%S')
    standard_df.loc[pd.to_datetime(standard_df['original_timestamp'], errors='coerce').isna(), 'timestamp'] = pd.NA

standard_df['confidence'] = 1.0
standard_df['detection_index'] = standard_df.groupby('image_name').cumcount() + 1

ordered_columns = [
    'image_name',
    'timestamp',
    'original_timestamp',
    'detection_index',
    'label',
    'confidence',
    'bbox_x0',
    'bbox_y0',
    'bbox_x1',
    'bbox_y1',
    'temp_mean_c',
    'temp_max_c',
    'image_exists',
    'image_timestamp_raw',
    'image_timestamp_source',
    'image_timestamp_error',
]
standard_df = standard_df[ordered_columns]

standard_df.to_csv(STANDARDIZED_CSV, index=False)
print(f'wrote: {STANDARDIZED_CSV}')
display(standard_df.head())


wrote: /Users/pck/TUe_AI_ES/2025_Q3/Team_Intership/2026_group_cleanroom/tmp/v2/input/v2_thermal_bbox_standardized.csv


,image_name,timestamp,original_timestamp,detection_index,label,confidence,bbox_x0,bbox_y0,bbox_x1,bbox_y1,temp_mean_c,temp_max_c,image_exists,image_timestamp_raw,image_timestamp_source,image_timestamp_error
0,IR_63142.jpg,2025-05-01 14:19:21,2025/5/1 15:19,1,Machine,1.0,461,298,483,318,21.886790,25.424925,True,2025:05:01 14:19:21,exif_306,<NA>
1,IR_63142.jpg,2025-05-01 14:19:21,2025/5/1 15:19,2,Machine,1.0,467,256,491,280,22.069757,25.502478,True,2025:05:01 14:19:21,exif_306,<NA>
2,IR_63142.jpg,2025-05-01 14:19:21,2025/5/1 15:19,3,Machine,1.0,404,248,438,271,21.645502,23.766699,True,2025:05:01 14:19:21,exif_306,<NA>
3,IR_63142.jpg,2025-05-01 14:19:21,2025/5/1 15:19,4,Machine,1.0,416,212,435,230,22.321835,25.575266,True,2025:05:01 14:19:21,exif_306,<NA>
4,IR_63142.jpg,2025-05-01 14:19:21,2025/5/1 15:19,5,Machine,1.0,452,207,507,248,23.885866,25.676907,True,2025:05:01 14:19:21,exif_306,<NA>


In [7]:
# check timestamp replacement and missing-image cases before projection.
timestamp_check = {
    'rows': len(standard_df),
    'rows_with_final_timestamp': int(standard_df['timestamp'].notna().sum()),
    'rows_without_final_timestamp': int(standard_df['timestamp'].isna().sum()),
    'rows_with_original_timestamp': int(standard_df['original_timestamp'].notna().sum()),
    'unique_images_missing': int((~timestamp_lookup['image_exists'].fillna(False)).sum()) if 'timestamp_lookup' in globals() else None,
}
display(timestamp_check)
display(standard_df[standard_df['timestamp'].isna()].head())


{'rows': 237515,
 'rows_with_final_timestamp': 146367,
 'rows_without_final_timestamp': 91148,
 'rows_with_original_timestamp': 237515,
 'unique_images_missing': 5543}

,image_name,timestamp,original_timestamp,detection_index,label,confidence,bbox_x0,bbox_y0,bbox_x1,bbox_y1,temp_mean_c,temp_max_c,image_exists,image_timestamp_raw,image_timestamp_source,image_timestamp_error
82572,IR_60327.jpg,NaN,2025/4/29 07:58,1,Light,1.0,472,10,570,41,24.463905,26.100000,False,NaN,NaN,<NA>
82573,IR_60327.jpg,NaN,2025/4/29 07:58,2,Screen,1.0,280,297,342,330,24.763021,24.923449,False,NaN,NaN,<NA>
114304,IR_57326.jpg,NaN,23:38.0,1,Person,1.0,192,222,447,480,29.645952,33.420715,False,NaN,NaN,<NA>
114305,IR_57326.jpg,NaN,23:38.0,2,Machine,1.0,461,234,485,259,23.230430,23.603697,False,NaN,NaN,<NA>
114306,IR_57326.jpg,NaN,23:38.0,3,Machine,1.0,99,201,125,253,27.282843,27.831585,False,NaN,NaN,<NA>


In [8]:
# run projection on the standardized bbox rows.
projection_args = SimpleNamespace(
    input_csv=STANDARDIZED_CSV,
    output_csv=PROJECTED_CSV,
    human_model_path=config_v2.HUMAN_PROJECTION_MODEL_PATH,
    machine_model_path=config_v2.MACHINE_PROJECTION_MODEL_PATH,
    normalized_factor=config_v2.NORMALIZED_FACTOR,
)

projected_rows = run_projection(projection_args)
len(projected_rows), projected_rows[:2]


Projection: 100%|██████████| 237515/237515 [00:09<00:00, 24983.11object/s]


(237515,
 [{'image_name': 'IR_63142.jpg',
   'timestamp': '2025-05-01 14:19:21',
   'original_timestamp': '2025/5/1 15:19',
   'detection_index': '1',
   'label': 'Machine',
   'confidence': '1.0',
   'bbox_x0': '461',
   'bbox_y0': '298',
   'bbox_x1': '483',
   'bbox_y1': '318',
   'temp_mean_c': '21.88679',
   'temp_max_c': '25.424925',
   'image_exists': 'True',
   'image_timestamp_raw': '2025:05:01 14:19:21',
   'image_timestamp_source': 'exif_306',
   'image_timestamp_error': '',
   'projected_x': '7.327785',
   'projected_y': '4.141180',
   'projection_model': 'machine'},
  {'image_name': 'IR_63142.jpg',
   'timestamp': '2025-05-01 14:19:21',
   'original_timestamp': '2025/5/1 15:19',
   'detection_index': '2',
   'label': 'Machine',
   'confidence': '1.0',
   'bbox_x0': '467',
   'bbox_y0': '256',
   'bbox_x1': '491',
   'bbox_y1': '280',
   'temp_mean_c': '22.069757',
   'temp_max_c': '25.502478',
   'image_exists': 'True',
   'image_timestamp_raw': '2025:05:01 14:19:21',
   '

In [9]:
# preview the projected output and class distribution.
projected_df = pd.read_csv(PROJECTED_CSV)
print(f'wrote: {PROJECTED_CSV}')
display(projected_df.head())
display(projected_df['label'].value_counts(dropna=False))
display(projected_df[['projected_x', 'projected_y']].describe())

wrote: /Users/pck/TUe_AI_ES/2025_Q3/Team_Intership/2026_group_cleanroom/tmp/v2/projection/v2_projected_detections.csv


/var/folders/r7/6cnjjpl12cl8kc68cxt7dd9w0000gn/T/ipykernel_42817/4246361609.py:2: DtypeWarning: Columns (0: timestamp, 1: image_timestamp_raw, 2: image_timestamp_source) have mixed types. Specify dtype option on import or set low_memory=False.
  projected_df = pd.read_csv(PROJECTED_CSV)


,image_name,timestamp,original_timestamp,detection_index,label,confidence,bbox_x0,bbox_y0,bbox_x1,bbox_y1,temp_mean_c,temp_max_c,image_exists,image_timestamp_raw,image_timestamp_source,image_timestamp_error,projected_x,projected_y,projection_model
0,IR_63142.jpg,2025-05-01 14:19:21,2025/5/1 15:19,1,Machine,1.0,461,298,483,318,21.886790,25.424925,True,2025:05:01 14:19:21,exif_306,NaN,7.327785,4.141180,machine
1,IR_63142.jpg,2025-05-01 14:19:21,2025/5/1 15:19,2,Machine,1.0,467,256,491,280,22.069757,25.502478,True,2025:05:01 14:19:21,exif_306,NaN,6.265264,4.480132,machine
2,IR_63142.jpg,2025-05-01 14:19:21,2025/5/1 15:19,3,Machine,1.0,404,248,438,271,21.645502,23.766699,True,2025:05:01 14:19:21,exif_306,NaN,5.582700,3.847756,machine
3,IR_63142.jpg,2025-05-01 14:19:21,2025/5/1 15:19,4,Machine,1.0,416,212,435,230,22.321835,25.575266,True,2025:05:01 14:19:21,exif_306,NaN,5.051721,4.170983,machine
4,IR_63142.jpg,2025-05-01 14:19:21,2025/5/1 15:19,5,Machine,1.0,452,207,507,248,23.885866,25.676907,True,2025:05:01 14:19:21,exif_306,NaN,5.635340,5.179559,machine


label
Machine    136806
Light       62510
Screen      16078
Window      12809
Person       9312
Name: count, dtype: int64

,projected_x,projected_y
count,237515.000000,237515.000000
mean,6.555326,2.962073
std,4.535691,2.003910
min,0.164745,0.046241
25%,3.522390,1.098652
50%,5.614198,3.501784
75%,7.880568,4.498605
max,25.281342,7.765507
